# RL

# Setup

In [ ]:
from thesis_project import (
    config,
)

config.default()

TICKER: config.FutTicker = "fbtp"

# Lobs directories
LOB_DIR = config.DATA_RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "fut"
LOB_CTD_DIR = LOB_DIR / "ctd"

# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.RES_FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.RES_TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_EXP_DIR = config.RES_EXP_DIR / TICKER / NOTEBOOK
    OUT_EXP_DIR.mkdir(parents=True, exist_ok=True)

# Common

# Main

## Baselines

In [ ]:
from thesis_project import rl
from thesis_project.dataset.features_market import FeatureSpec
from thesis_project.dataset.features_transforms import Identity

market: list[FeatureSpec] = [
    FeatureSpec(base_id="basis", transforms=[Identity()]),
]

dataset_config = rl.dataset.DatasetConfig(
    ticker="fbtp", n_jobs=1, contract_mode="round", market=market
)

env_config_serial = rl.env.EnvConfig(
    reset_mode="serial",
    price_mode="quoted",
    persistence_min=10,
    trajectory_min=0,
)

dummy_dataset = rl.dataset.build_rl_dataset(dataset_config)

env = rl.train.make_env(dataset=dummy_dataset, env_config=env_config_serial)


rl.evaluation.evaluate_model()

## Report

In [ ]:
import json

from thesis_project.rl import report

root = config.RES_EXP_DIR / TICKER / "rl"
run = 3
folder = f"run-{run:03d}"
path = root / folder

with open(path / "config.json", "r") as f:
    settings = json.load(f)

seeds = settings["seeds"]

In [ ]:
records = report.load_records(
    path,
    batch_size=512,
    clip_range=0.3,
    seeds=seeds,
    split="test",
)

report.report(records)